返回的是字典，类型不匹配的时候不抛出异常

@dataclass是python标准库dataclasses提供的类装饰器，用于简化"以字段为核心"的数据类的定义

加上@dataclass后，会根据字段声明自动生成常用方法，__init__负责初始化  __repr__负责控制输出格式  __eq__负责控制相应行为

行为上接近于手写__init__ __repr__ __eq__的普通类，但是并不完全等价，@dataclass修饰后的类还是普通类，被python标准库标记为数据类

In [1]:
from dataclasses import dataclass

from pydantic import BaseModel, Field, ValidationError
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
from typing import Literal, Optional
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate


#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
message=[HumanMessage("我想知道胡歌的一些信息")]
@dataclass
class Person():
    """
    人物信息
    """
    name: str=Field(description="姓名",min_length=2,max_length=20)
    age:int=Field(description="年龄",gt=0,lt=120)
    occupation:str=Field(description="职业")
    hobby:str=Field(description="爱好")
    height:int=Field(description="身高",gt=0,lt=300)
    gender:str=Field(description="性别")
    weight:int=Field(description="体重",gt=0)
structured_model=model.with_structured_output(Person)

result=structured_model.invoke(message)
print(result)
# rprint(type(result))


{'name': '胡歌', 'age': 43, 'occupation': '演员、歌手', 'hobby': '摄影、骑行、阅读', 'height': 185, 'gender': '男', 'weight': 75}
